In [1]:
import os, sys 
from pathlib import Path
from os.path import dirname, realpath
script_dir = Path(dirname(realpath('.')))
module_dir = str(script_dir)
sys.path.insert(0, module_dir + '/modules')
import numpy as np
import utility as ut
import matplotlib.pyplot as plt
import pandas as pd
import warnings
from scipy import stats
import seaborn as sns
import sample as sm
import chain as arch 
import torch
# warnings.filterwarnings('ignore')

In [2]:
Uo = np.load('../data/L63-trajectories/train.npy')[:, :20000]
Vo = np.load('../data/L63-trajectories/test.npy')
L0, L1 = 0.4, 3.5
D, D_r, B = 3, 256, 2
beta = 4e-5
m = 50

In [3]:
B = 1
rf1 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf1.init()
tau1 = rf1.compute_tau_f_(Vo[0:m])

Time taken by sample is 0.0047 seconds
Time taken by compute_tau_f_ is 1.0533 seconds


In [4]:
B = 2
rf2 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf2.init()
tau2 = rf2.compute_tau_f_(Vo[0:m])

Time taken by sample is 0.0046 seconds
Time taken by sample is 0.0049 seconds
Time taken by compute_tau_f_ is 1.8101 seconds


In [5]:
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

B = 3
rf3 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf3.init()
tau3 = rf3.compute_tau_f_(Vo[0:m])

Time taken by sample is 0.0046 seconds
Time taken by sample is 0.0090 seconds
Time taken by sample is 0.0136 seconds
Time taken by compute_tau_f_ is 2.6024 seconds


In [6]:
B = 10
rf10 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf10.init()
tau10 = rf10.compute_tau_f_(Vo[0:m])

Time taken by sample is 0.0046 seconds
Time taken by sample is 0.0064 seconds
Time taken by sample is 0.0048 seconds
Time taken by sample is 0.0075 seconds
Time taken by sample is 0.0065 seconds
Time taken by sample is 0.0109 seconds
Time taken by sample is 0.0099 seconds
Time taken by sample is 0.0048 seconds
Time taken by sample is 0.0075 seconds
Time taken by sample is 0.0049 seconds
Time taken by compute_tau_f_ is 8.1104 seconds


In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)
sns.histplot(tau1[1], ax=ax, label=r'depth=1, $\mathbb{E}[\tau_f]$'+f'={tau1[1].mean():.2f}', alpha=0.7, stat='probability')
sns.histplot(tau2[1], ax=ax, label=r'depth=2, $\mathbb{E}[\tau_f]$'+f'={tau2[1].mean():.2f}', alpha=0.7, stat='probability')
sns.histplot(tau3[1], ax=ax, label=r'depth=3, $\mathbb{E}[\tau_f]$'+f'={tau3[1].mean():.2f}', alpha=0.7, stat='probability')
sns.histplot(tau10[1], ax=ax, label=r'depth=10, $\mathbb{E}[\tau_f]$'+f'={tau10[1].mean():.2f}', alpha=0.7, stat='probability')
ax.legend()
ax.set_title(fr'$\tau_f$, architecture=chain, $D_r$={D_r}')
plt.savefig(f'../data/plots/chain-tau_f-D_r-{D_r}.png', bbox_inches='tight', dpi=300)

In [ ]:
rf3.net.inner[1].register_forward_hook(get_activation('inner.1'))
x = torch.from_numpy(Uo[:, 0])
output = rf3.net(x)
0.4<torch.abs(activation['inner.1']) 

In [ ]:
Uo[:, 0].shape

In [ ]:
for name, m in rf3.net.named_modules():
    print(name)

In [ ]:
rf3.net.inner.register_forward_hook(get_activation('inner'))

In [8]:
rf1.loss_wo_reg()

tensor(0.1326, grad_fn=<SumBackward0>)

In [7]:
rf2.loss_wo_reg()

tensor(0.2263, grad_fn=<SumBackward0>)

In [6]:
tau1[1]

array([3.367 , 4.2224, 3.6036, 2.8574, 5.0414, 8.1536, 4.2042, 4.0586,
       4.1678, 6.3154, 3.1486, 4.0222, 4.641 , 3.3852, 2.9484, 3.9494,
       3.8038, 2.8938, 4.368 , 5.3508, 3.822 , 4.6228, 3.2578, 3.822 ,
       4.2406, 3.4216, 5.369 , 4.2042, 4.0404, 5.1324, 2.5662, 4.8594,
       2.4934, 2.821 , 3.64  , 3.0758, 2.2204, 3.731 , 5.6966, 5.7512,
       2.8028, 2.4024, 5.9514, 2.5662, 2.3296, 4.7502, 4.4408, 3.1668,
       2.1294, 3.5126])

In [ ]:
rf1.std

In [ ]:
rf1.mean

In [ ]:
Uo[:, :].T.mean(axis=0)

In [ ]:
Uo.T.shape

In [14]:
0.69*1500/60

17.25

In [11]:
torch.cuda.is_available()

False

In [13]:
torch.mps.get_device_name(0)

AttributeError: module 'torch.mps' has no attribute 'get_device_name'

In [7]:
rf3.train(epochs=1200, learning_rate=1e-4, batch_size=200, update_fraction=0.1, update_threshold=-1e-4)

epoch=0.000	loss=4.610	change=nan	lr=0.000	time=0.098	
epoch=1.000	loss=0.006	change=-0.999	lr=0.000	time=0.099	
epoch=2.000	loss=0.003	change=-0.446	lr=0.000	time=0.092	
epoch=3.000	loss=0.003	change=0.003	lr=0.000	time=0.094	
epoch=4.000	loss=0.008	change=1.468	lr=0.000	time=0.092	
epoch=5.000	loss=0.012	change=0.500	lr=0.000	time=0.091	
epoch=6.000	loss=0.003	change=-0.755	lr=0.000	time=0.091	
epoch=7.000	loss=0.014	change=3.598	lr=0.000	time=0.092	
epoch=8.000	loss=0.005	change=-0.601	lr=0.000	time=0.091	
epoch=9.000	loss=0.019	change=2.511	lr=0.000	time=0.092	
epoch=10.000	loss=0.007	change=-0.638	lr=0.000	time=0.093	
epoch=11.000	loss=0.006	change=-0.199	lr=0.000	time=0.091	
epoch=12.000	loss=0.020	change=2.479	lr=0.000	time=0.091	
epoch=13.000	loss=0.006	change=-0.716	lr=0.000	time=0.090	
epoch=14.000	loss=0.007	change=0.290	lr=0.000	time=0.089	
epoch=15.000	loss=0.005	change=-0.273	lr=0.000	time=0.090	
epoch=16.000	loss=0.003	change=-0.333	lr=0.000	time=0.091	
epoch=17.000	loss

In [8]:
tau3 = rf3.compute_tau_f_(Vo[0:m])

Time taken by compute_tau_f_ is 2.5377 seconds


In [9]:
tau3

(array([ 6.8068,  9.0454,  7.2072,  7.7168,  5.8786,  6.9524,  7.3892,
         9.1364,  7.1344,  6.1334,  6.0788,  7.5712,  6.2426, 11.1202,
        10.2284,  4.2588,  3.9858,  6.3882,  4.6046,  5.7694,  6.1516,
         8.2264,  6.9706,  5.4236,  9.1364,  8.5358,  7.5894,  9.7916,
         4.2952,  5.3326,  5.9332,  6.4428,  2.6754,  3.2214,  8.7906,
         8.8816,  2.275 ,  8.2992,  5.9514,  7.9534,  5.7148,  8.5176,
         6.1698,  6.006 ,  5.733 ,  6.4974,  9.555 ,  5.8058,  9.3366,
         3.8402]),
 array([4.7502, 6.3154, 5.642 , 6.2426, 4.4226, 4.1496, 6.9342, 8.8634,
        5.6784, 3.6764, 3.1486, 7.371 , 6.006 , 9.5914, 9.0454, 4.0404,
        3.822 , 6.1698, 4.3316, 3.4034, 3.8584, 6.6794, 6.006 , 3.8948,
        7.644 , 6.097 , 5.46  , 7.0434, 4.1314, 5.0778, 2.6754, 4.9504,
        2.5298, 2.9302, 8.2264, 8.463 , 2.0748, 3.822 , 5.7512, 5.8422,
        5.3508, 7.007 , 5.9514, 5.7876, 5.5146, 6.097 , 8.0262, 3.1122,
        7.7896, 3.6036]),
 array([0.74575557, 0.4491

In [23]:
rf3.Y

tensor([[ -9.3860, -14.6547,  19.6255],
        [-10.4542, -15.8231,  21.5584],
        [-11.5119, -16.6562,  23.9250],
        ...,
        [ -8.5377,  -4.8378,  31.2825],
        [ -7.8119,  -4.2816,  30.3809],
        [ -7.1350,  -3.9159,  29.3969]])